In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# Put project root on sys.path so "source" is importable
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent  # notebooks/ -> root/
print(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

/home/mpominova/lcms-foundation-model


In [ ]:
import os
import yaml
import numpy as np
import pandas as pd
import polars as pl

import torch
import torch.nn as nn
import pytorch_lightning as L
from tqdm import tqdm
from depthcharge.data import SpectrumDataset, spectra_to_df, preprocessing
from torch.utils.data import DataLoader

from source.dataset import LanceMapDataset, RunDataset
from source.model import MS1Encoder
from source.scheduler import CosineWarmupScheduler
from source.config import ExperimentConfig, DataConfig, ModelConfig, OptimizerConfig, TrainingConfig

from IPython.display import clear_output

/home/mpominova/.local/share/mamba/envs/lcms-foundation/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def load_config(config_path):
    """Load configuration from YAML file."""
    with open(config_path, "r") as f:
        config_dict = yaml.safe_load(f)
        config = ExperimentConfig(
            name=config_dict['name'],
            data=DataConfig(**config_dict['data']),
            model=ModelConfig(**config_dict['model']),
            optimizer=OptimizerConfig(**config_dict['optimizer']),
            training=TrainingConfig(**config_dict['training'])
        )
    return config

In [5]:
# Load training data
data_root_dir = "/mnt/data/shared/lc_ms_foundation/"
dset_name = "abele_data"
data_dir = os.path.join(data_root_dir, dset_name, "mzml")

mzml_files = os.listdir(data_dir)
print("Total N files:", len(mzml_files))

Total N files: 1334


In [6]:
# meta_df = pl.read_csv("../data/filtered_abele_metadata.csv")
meta_df = pd.read_csv(os.path.join(
    data_root_dir, 
    dset_name,
    "all_abele_metadata.csv"
))

meta_df = meta_df.rename({
    "characteristics[organism]": "organism",
    "comment[data file]": "data_file"
}, axis=1)
meta_df

,organism,genus,data_file
0,Bacillus cereus,Bacillus,BBM_429_P110_31_MIA_007
1,Lacticaseibacillus paracasei,Lacticaseibacillus,BBM_429_P110_31_MIA_022
2,Shigella sonnei,Shigella,BBM_437_P110_31_MIA_026
3,Pantoea cypripedii,Pantoea,BBM_437_P110_31_MIA_028
4,Hafnia alvei,Hafnia,BBM_437_P110_31_MIA_029
...,...,...,...
1389,Escherichia coli,Escherichia,BBM_749_P110_38_MIA_095
1390,Escherichia coli,Escherichia,BBM_749_P110_38_MIA_096
1391,Pseudomonas aeruginosa,Pseudomonas,BBM_750_P110_38_MIA_001
1392,Pseudomonas aeruginosa,Pseudomonas,BBM_750_P110_38_MIA_002


In [7]:
genus_counts = meta_df["genus"].value_counts()
genus_counts[:20]

genus
Pseudomonas           312
Staphylococcus        136
Bacillus              109
food                   60
Escherichia            51
Enterococcus           42
Serratia               27
Acinetobacter          24
Lactococcus            21
Buttiauxella           21
Leuconostoc            18
Sphingobacterium       18
Bifidobacterium        18
Ligilactobacillus      15
Lacticaseibacillus     15
Micrococcus            15
Listeria               15
Klebsiella             12
Lysinibacillus         12
Weissella              12
Name: count, dtype: int64

In [11]:
meta_df[meta_df["genus"] == "Enterococcus"]["organism"].value_counts()

organism
Enterococcus hirae              3
Enterococcus raffinosus         3
Enterococcus avium              3
Enterococcus faecium            3
Enterococcus mundtii            3
Enterococcus casseliflavus      3
Enterococcus saccharolyticus    3
Enterococcus cecorum            3
Enterococcus malodoratus        3
Enterococcus faecalis           3
Enterococcus durans             3
Enterococcus sulfureus          3
Enterococcus columbae           3
Enterococcus dispar             3
Name: count, dtype: int64

In [10]:
meta_df[meta_df["genus"] == "Serratia"]["organism"].value_counts()

organism
Serratia fonticola         3
Serratia liquefaciens      3
Serratia proteamaculans    3
Serratia ficaria           3
Serratia plymuthica        3
Serratia odorifera         3
Serratia marcescens        3
Serratia grimesii          3
Serratia rubidaea          3
Name: count, dtype: int64

In [12]:
meta_df[meta_df["genus"] == "Acinetobacter"]["organism"].value_counts()

organism
Acinetobacter johnsonii         6
Acinetobacter junii             3
Acinetobacter haemolyticus      3
Acinetobacter lwoffii           3
Acinetobacter calcoaceticus     3
Acinetobacter radioresistens    3
Acinetobacter baumannii         3
Name: count, dtype: int64

In [8]:
genus_counts.sum()

1394

In [9]:
genus_counts.size

120

In [20]:
# before we had 9 + 21 for SSL_train
# and 15 / 15 for probe_train/probe_val

In [19]:
# here we have

# 1.
# - food - 60 -- we don't use
# - 3 genuses >100 -- can be used for SSL_train?
# - 2 genuses 40 < < 100 -- can be used for downstream? 

# UPD: we would like to split them so that 
# all runs from the same organism are in the same split (train/val)
# but Escherichia has very uneven organism distribution: 
# 48 coli vs 3 fergusonii, and coli samples cannot be split into smaller groups
# - so maybe it's better to choose a different genus as a second class?

# THEN
# 2. 
# - 3 genuses >100 + Escherichia -- for SSL train
# - next 2 genuses (Enterococcus + Serratia) -- for downstream
    # - if not enough data for downstream, add next (Acinetobacter)
    # and classify Enterococcus (42) vs Serratia + Acinetobacter (51)
    # or classify 3 classes

# and others could be just ignored for now and later added to SSL? 

In [ ]:
# for now, consider only runs that we plan to use

In [18]:
# 701 runs for SSL training
genus_counts[genus_counts > 21].sum() - 60

701

In [14]:
# 93 in total for 
genus_counts["Enterococcus"] + genus_counts["Serratia"]

69

In [15]:
selected_genuses = [
    "Pseudomonas", 
    "Staphylococcus", 
    "Bacillus", 
    "Escherichia", 
    "Enterococcus",
    "Serratia",
    "Acinetobacter",
]
meta_df = meta_df[meta_df["genus"].isin(selected_genuses)]
meta_df

,organism,genus,data_file
0,Bacillus cereus,Bacillus,BBM_429_P110_31_MIA_007
10,Pseudomonas aeruginosa,Pseudomonas,BBM_437_P110_31_MIA_036
12,Bacillus pumilus,Bacillus,BBM_429_P110_31_MIA_023
13,Serratia fonticola,Serratia,BBM_437_P110_31_MIA_040
17,Staphylococcus nepalensis,Staphylococcus,BBM_441_P110_31_MIA_001
...,...,...,...
1389,Escherichia coli,Escherichia,BBM_749_P110_38_MIA_095
1390,Escherichia coli,Escherichia,BBM_749_P110_38_MIA_096
1391,Pseudomonas aeruginosa,Pseudomonas,BBM_750_P110_38_MIA_001
1392,Pseudomonas aeruginosa,Pseudomonas,BBM_750_P110_38_MIA_002


In [16]:
meta_df["genus"].value_counts()

genus
Pseudomonas       312
Staphylococcus    136
Bacillus          109
Escherichia        51
Enterococcus       42
Serratia           27
Acinetobacter      24
Name: count, dtype: int64

In [19]:
# I will use Pseudomonas, Staphylococcus, Bacillus, Escherichia for petraining
# and split Enterococcus and Serratia between train and val for downstream
# - group by organism, put all files of the same organism in one split

In [21]:
meta_df[meta_df["genus"] == "Enterococcus"]["organism"].value_counts()

organism
Enterococcus hirae              3
Enterococcus raffinosus         3
Enterococcus avium              3
Enterococcus faecium            3
Enterococcus mundtii            3
Enterococcus casseliflavus      3
Enterococcus saccharolyticus    3
Enterococcus cecorum            3
Enterococcus malodoratus        3
Enterococcus faecalis           3
Enterococcus durans             3
Enterococcus sulfureus          3
Enterococcus columbae           3
Enterococcus dispar             3
Name: count, dtype: int64

In [ ]:
# for learning curves, we will need several balanced subsets that gradually increase SSL_train size
# this means, we also need to take into account
# - genus - each subset should contain samples from each of 3 training genuses 
    # (for now - later we can also experiment with how number of different genuses in train impacts performance)

# - maybe gradient length -- each subset should include all gradient lengths of the same sample
# -> so we will again need to split them by sample?

In [14]:
def get_gradient_lenght(run_name):
    run_name = run_name.split("_")
    if len(run_name) < 7:
        return ""
    return run_name[6]

In [22]:
meta_df["peak_file"] = meta_df["data_file"] + ".mzML"
meta_df

/tmp/ipykernel_18155/1296095814.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  meta_df["peak_file"] = meta_df["data_file"] + ".mzML"


,organism,genus,data_file,peak_file
0,Bacillus cereus,Bacillus,BBM_429_P110_31_MIA_007,BBM_429_P110_31_MIA_007.mzML
10,Pseudomonas aeruginosa,Pseudomonas,BBM_437_P110_31_MIA_036,BBM_437_P110_31_MIA_036.mzML
12,Bacillus pumilus,Bacillus,BBM_429_P110_31_MIA_023,BBM_429_P110_31_MIA_023.mzML
13,Serratia fonticola,Serratia,BBM_437_P110_31_MIA_040,BBM_437_P110_31_MIA_040.mzML
17,Staphylococcus nepalensis,Staphylococcus,BBM_441_P110_31_MIA_001,BBM_441_P110_31_MIA_001.mzML
...,...,...,...,...
1389,Escherichia coli,Escherichia,BBM_749_P110_38_MIA_095,BBM_749_P110_38_MIA_095.mzML
1390,Escherichia coli,Escherichia,BBM_749_P110_38_MIA_096,BBM_749_P110_38_MIA_096.mzML
1391,Pseudomonas aeruginosa,Pseudomonas,BBM_750_P110_38_MIA_001,BBM_750_P110_38_MIA_001.mzML
1392,Pseudomonas aeruginosa,Pseudomonas,BBM_750_P110_38_MIA_002,BBM_750_P110_38_MIA_002.mzML


In [23]:
dset_mzml_files = meta_df["peak_file"].to_list()
len(dset_mzml_files)

701

In [24]:
config = load_config("../config.yaml")
config

ExperimentConfig(name='ms1-mz-200_peaks-sqrt-clf_run_retrain', data=DataConfig(train_dir='train_mzml', val_dir='val_mzml', batch_size=2000, max_num_peaks=200), model=ModelConfig(d_model=256, nhead=8, dim_feedforward=512, n_layers=6, dropout=0.1, n_bins=1200, bin_mz_min=300, bin_mz_max=1500, masked_peaks_fraction=0.3), optimizer=OptimizerConfig(lr=0.0005, warmup_iters=1000, cosine_schedule_period_iters=64000), training=TrainingConfig(checkpoint_path='./train_checkpoints', max_epochs=1000, gradient_clip_val=5, accelerator='gpu', devices=1))

In [25]:
preprocessing_fn = [
    preprocessing.filter_intensity(max_num_peaks=config.data.max_num_peaks),
    preprocessing.scale_intensity(scaling="root", max_intensity=1.),
]

In [26]:
data_dir

'/mnt/data/shared/lc_ms_foundation/abele_data/mzml'

In [ ]:
# filter MS1 and save to parquet for quicker loading later

# it may be also a good idea to save them all together? a single large parquet file? 
# but maybe just do separately for now

In [27]:
save_dir = "/mnt/data/mpominova/lcms_foundation_data/abele_data"

for mzml_file in tqdm(dset_mzml_files):
    save_file = os.path.splitext(mzml_file)[0] + ".parquet"
    save_path = os.path.join(save_dir, save_file)
    if os.path.exists(save_path):
        print(f"{save_path} already exists - skipped.")
        continue
    
    df = spectra_to_df(
        os.path.join(data_dir, mzml_file),
        metadata_df=None,
        ms_level=1, # only extract MS1
        preprocessing_fn=preprocessing_fn, # here we also take top-n peaks and transform intensity -- do we want it now or not? 
        valid_charge=None,
        custom_fields=None,
        progress=True,
    )
    df = df[["peak_file", "mz_array", "intensity_array"]]
    df.write_parquet(save_path)
    clear_output(True)

100%|██████████| 701/701 [23:19<00:00,  2.00s/it]

/mnt/data/mpominova/lcms_foundation_data/abele_data/BBM_436_P110_31_MIA_005_10.parquet already exists - skipped.
/mnt/data/mpominova/lcms_foundation_data/abele_data/BBM_436_P110_31_MIA_014_10.parquet already exists - skipped.
/mnt/data/mpominova/lcms_foundation_data/abele_data/BBM_436_P110_31_MIA_024_10.parquet already exists - skipped.
/mnt/data/mpominova/lcms_foundation_data/abele_data/BBM_338_P110_28_DDA_002.parquet already exists - skipped.
/mnt/data/mpominova/lcms_foundation_data/abele_data/BBM_338_P110_28_DDA_011.parquet already exists - skipped.
/mnt/data/mpominova/lcms_foundation_data/abele_data/BBM_338_P110_28_DDA_012.parquet already exists - skipped.
/mnt/data/mpominova/lcms_foundation_data/abele_data/BBM_338_P110_28_DDA_014.parquet already exists - skipped.
/mnt/data/mpominova/lcms_foundation_data/abele_data/BBM_338_P110_28_DDA_015.parquet already exists - skipped.
/mnt/data/mpominova/lcms_foundation_data/abele_data/BBM_338_P110_28_DDA_016.parquet already exists - skipped.
/